# 02 — Silver: Cleaning & Transforms

**Tickets:** I-03, I-04, I-05  
**Purpose:** Clean Bronze data into a reliable Silver table — cast types, drop corrupt rows, handle missing values, add derived columns.

---

## Setup

In [ ]:
from src.constants import BRONZE_TABLE
from src.transforms import (
    add_zone_bins,
    cast_datetime_columns,
    cast_numeric_columns,
    clean_gps_coordinates,
    deduplicate,
    drop_corrupt_rows,
    recover_rate_code_id,
    standardise_column_names,
)

print("Setup complete")

## Read Bronze table

In [ ]:
bronze_df = spark.read.table(BRONZE_TABLE)
print(f"Bronze rows: {bronze_df.count():,}")
print(f"Bronze columns: {bronze_df.columns}")
bronze_df.printSchema()

## I-03 — Column name standardisation & type casting

In [ ]:
bronze_count = bronze_df.count()

# I-03a — Standardise column names (VendorID -> vendor_id, RateCodeID -> rate_code_id)
silver_df = standardise_column_names(bronze_df)

# I-03b — Recover rate_code_id from _rescued_data JSON (Finding #1: 73% of rows affected)
silver_df = recover_rate_code_id(silver_df)

# I-03c — Cast datetime columns to TimestampType
silver_df = cast_datetime_columns(silver_df)

# I-03d — Cast numeric columns to correct types (int / double)
silver_df = cast_numeric_columns(silver_df)

# I-03e — Deduplicate exact duplicate rows (Finding #9: 772 rows)
silver_df = deduplicate(silver_df)

# I-03f — Drop structurally corrupt rows (nulls in required fields)
silver_df = drop_corrupt_rows(silver_df)

# I-03g — Clean GPS coordinates: NULL out (0,0) and out-of-NYC-bbox (Finding #10)
silver_df = clean_gps_coordinates(silver_df)

# I-03h — Add grid-binned zone columns from cleaned lat/lon (needed by Gold: BQ-1, BQ-2, A-02, A-03)
silver_df = add_zone_bins(silver_df)

after_count = silver_df.count()
print(f"Bronze rows:  {bronze_count:,}")
print(f"Silver rows:  {after_count:,}")
print(
    f"Dropped:      {bronze_count - after_count:,} ({(bronze_count - after_count) / bronze_count * 100:.2f}%)"
)
silver_df.printSchema()
display(silver_df.limit(5))

## I-04 — Handle missing values & outliers

In [ ]:
# TODO: drop_zero_distance_trips(df)
# TODO: drop_invalid_fares(df)
# TODO: handle any remaining nulls

## I-05 — Derived columns

In [ ]:
# TODO: add trip_duration_min, hour_of_day, day_of_week, is_weekend

## Write Silver Delta table

In [ ]:
# TODO: write to Silver Delta table